In [19]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.


In [20]:
import os
import json
import re
from typing import List, Dict, Any
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from groq import Groq
from transformers import pipeline
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG & MODELS
# ─────────────────────────────────────────────

app = FastAPI(title="Indy ADHD Copilot API")

# Initialize Models once on startup
print("Loading NLP model...")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)
LLM_MODEL = "llama-3.1-8b-instant"
SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]  # List of {"role": "user/assistant", "content": "..."}

# ─────────────────────────────────────────────
# LOGIC HELPERS
# ─────────────────────────────────────────────

def score_text(text: str):
    candidate_labels = list(LABEL_DESCRIPTIONS.values())
    result = nlp_classifier(text, candidate_labels, multi_label=False)
    
    scores = {}
    for label, score in zip(result['labels'], result['scores']):
        for clean_name, desc in LABEL_DESCRIPTIONS.items():
            if label == desc:
                scores[clean_name] = round(score, 4)
    
    dominant = max(scores, key=scores.get)
    return {"scores": scores, "dominant": dominant}

def get_past_context(session_id: int):
    # Same logic as your script: look for session_001.json etc.
    context_blocks = []
    for i in range(session_id - 1, max(0, session_id - 4), -1):
        path = SESSIONS_DIR / f"session_{i:03d}.json"
        if path.exists():
            with open(path, "r") as f:
                s = json.load(f)
                block = f"Session {s['session_id']}: Mood: {s.get('mood')}, Wins: {s.get('small_wins')}"
                context_blocks.append(block)
    return "\n".join(context_blocks) if context_blocks else "No history."

def load_or_create_session(session_id: int) -> dict:
    """Load session or create new one if doesn't exist"""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    else:
        return {
            "session_id": session_id,
            "created": datetime.now().isoformat(),
            "conversation": [],
            "cognitive_shifts": []
        }

def save_session(session_id: int, session_data: dict) -> None:
    """Save session to file"""
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

# ─────────────────────────────────────────────
# ENDPOINTS
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. Cognitive Shift Analysis (The 'NLP' Layer)
        analysis = score_text(req.message)
        
        # 2. Get RAG Context
        past_context = get_past_context(req.session_id)
        
        # 3. Construct System Prompt
        system_prompt = f"""You are Indy, an ADHD copilot. Be gentle, short, no guilt.
Respond in the same language as the user.
PAST CONTEXT: {past_context}
After your reply, PLEASE FOR THE LOVE OF GOD ALWAYS end with exactly: ### UPDATED_NOTES {{json}}"""

        # 4. Call Groq with full conversation history for context
        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        
        completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            temperature=0.7
        )
        
        raw_reply = completion.choices[0].message.content
        
        # 5. Parsing & Cleaning
        clean_text = raw_reply.split("### UPDATED_NOTES")[0].strip()
        
        # Extract JSON notes if present
        notes = {}
        if "### UPDATED_NOTES" in raw_reply:
            try:
                json_str = raw_reply.split("### UPDATED_NOTES")[-1].strip()
                json_str = re.sub(r"^```json|```$", "", json_str).strip()
                notes = json.loads(json_str)
            except:
                notes = {"error": "Failed to parse notes"}

        # 6. Save session with conversation history
        session_data = load_or_create_session(req.session_id)
        
        # Append user message to conversation
        session_data["conversation"].append({
            "role": "user",
            "content": req.message,
            "timestamp": datetime.now().isoformat()
        })
        
        # Append assistant response to conversation
        session_data["conversation"].append({
            "role": "assistant",
            "content": clean_text,
            "timestamp": datetime.now().isoformat(),
            "cognitive_shift": analysis,
            "notes": notes
        })
        
        # Track cognitive shifts for later analysis
        session_data["cognitive_shifts"].append({
            "message": req.message,
            "analysis": analysis,
            "timestamp": datetime.now().isoformat()
        })
        
        save_session(req.session_id, session_data)
        
        return {
            "reply": clean_text,
            "cognitive_shift": analysis,
            "extracted_notes": notes
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Loading NLP model...


In [ ]:
# Ensure nest_asyncio is available in the notebook environment so uvicorn can run
# without raising "asyncio.run() cannot be called from a running event loop".
# The %pip magic is appropriate inside a Jupyter cell.
%pip install -q nest_asyncio

import nest_asyncio
nest_asyncio.apply()

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

Note: you may need to restart the kernel to use updated packages.



You should consider upgrading via the 'c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\python.exe -m pip install --upgrade pip' command.
INFO:     Started server process [18336]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62278 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:62298 - "POST /chat HTTP/1.1" 200 OK
